# Performance Comparison of LLMs for Urdu Text Classification Using PEFT (LoRA / QLoRA / DoRA)
**Kaggle settings:** Accelerator = GPU T4 x2 | Internet = On | Add-ons > Secrets > label `LLMs_Label` (value = your HF read token, Attached ON)

**Run order: ALWAYS run the cells top to bottom (or use Run All).** Variables live only in the current session,
so after a restart/stop you must run the config cell again before any later cell (otherwise: `NameError: N_PER_CLASS`).
Switch `DRY_RUN` in the config cell: `True` = quick test, `False` = real experiments.

**Error -> which section to run (NameError: name 'X' is not defined):**
| Missing name | Run this section |
|---|---|
| `N_PER_CLASS`, `MODELS`, `METHODS`, `OUT`, `SEED`, `EPOCHS` | Section 2 |
| `df` | Section 4 |
| `train_df`, `val_df`, `test_df`, `classes`, `id2label` | Section 5 |
| `results`, `save_results`, `scores` | Section 6 |
| `run_experiment` | Section 8 |
| `summary` | Section 10 |

When you report an error, tell me the **Section number** (each section prints `=== Section N ===` first, so it is also visible in commit logs).

## Section 1 — Install packages
Needs: nothing (run first in a NEW session).

In [ ]:
print('=== Section 1: Install packages ===')
!pip install -q bitsandbytes==0.45.5 transformers==4.56.2 peft==0.15.2 accelerate==1.6.0
#!pip uninstall -y torchao

## Section 2 — Config + Hugging Face login
Needs: Section 1. Switch DRY_RUN here. Defines: N_PER_CLASS, EPOCHS, MODELS, METHODS, SEED, OUT, MAX_LEN, LR, RANK.

In [ ]:
import os; 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print('=== Section 2: Config + Hugging Face login ===')
import os, re, gc, json, time, random
import numpy as np, pandas as pd, torch
from huggingface_hub import login
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# HF login (needed for gated models: Llama, Gemma). The token stays in Kaggle Secrets, never in code.
SECRET_LABEL = "LLMs_Label"               # must match the label in Add-ons > Secrets
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret(SECRET_LABEL))
    print("HF login OK")
except Exception as e:
    print("HF login FAILED:", str(e)[:150])

DRY_RUN = False                            # True = quick test | False = real run
SEED, OUT, MAX_LEN, LR, RANK = 42, "/kaggle/working", 256, 2e-4, 16
if DRY_RUN:
    N_PER_CLASS, EPOCHS = 500, 1
    MODELS  = {"Qwen2.5-3B": "Qwen/Qwen2.5-3B", "Llama-3.2-3B": "meta-llama/Llama-3.2-3B"}     # three model are enough to test the pipeline
    METHODS = ["lora"]
else:
    N_PER_CLASS, EPOCHS = 2000, 2
    MODELS = {
        "Qwen2.5-3B":   "Qwen/Qwen2.5-3B",
        "Llama-3.2-3B": "meta-llama/Llama-3.2-3B",
        "Gemma-2-2B":   "google/gemma-2-2b",
    }
    METHODS = ["lora"]                    # commit 1: ["lora"] | commit 2: ["qlora"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"DRY_RUN={DRY_RUN} | N_PER_CLASS={N_PER_CLASS} EPOCHS={EPOCHS} MAX_LEN={MAX_LEN}")
print("MODELS:", list(MODELS), "| METHODS:", METHODS)

## Section 3 — Access check (HF + models)
Needs: Section 2. Optional: only checks the token and model access.

In [ ]:
print('=== Section 3: Access check (HF + models) ===')
# Optional access check (does not stop the notebook if something fails)
from huggingface_hub import whoami, hf_hub_download
try: print("HF user:", whoami()["name"])
except Exception as e: print("whoami failed:", type(e).__name__)
for name, hf_id in MODELS.items():
    try: hf_hub_download(hf_id, "config.json"); print("OK  ", hf_id)
    except Exception as e: print("FAIL", hf_id, type(e).__name__)

## Section 4 — Load dataset
Needs: Section 2. Defines: df.

In [ ]:
print('=== Section 4: Load dataset ===')
import glob
print(glob.glob("/kaggle/input/**/*.csv", recursive=True))
DATA_PATH = "/kaggle/input/datasets/abrar1515/urdu-news-dataset-1m/urdu-news-dataset-1M.csv"
TEXT_COL, LABEL_COL = "News Text", "Category"
df = pd.read_csv(DATA_PATH, usecols=[TEXT_COL, LABEL_COL])     # only 2 columns -> less RAM, faster
print(df.shape, f"{df.memory_usage(deep=True).sum() / 1e9:.1f} GB")
print(df[LABEL_COL].value_counts())
df.head(3)

## Section 5 — Clean + sample + split
Needs: Sections 2, 4. Defines: train_df, val_df, test_df, classes, label2id, id2label.

In [ ]:
print('=== Section 5: Clean + sample + split ===')
def clean_urdu(t):
    t = str(t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)                 # URLs
    t = re.sub(r"[\u064B-\u065F\u0670]", "", t)             # diacritics
    t = t.replace("ي", "ی").replace("ك", "ک").replace("ى", "ی")   # Arabic -> Urdu letter forms
    t = re.sub(r"[^\u0600-\u06FF0-9\s۔؟،!?.,]", " ", t)     # keep Urdu script, digits, punctuation
    return re.sub(r"\s+", " ", t).strip()

d = df[[TEXT_COL, LABEL_COL]].dropna().rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
d["label"] = d["label"].astype(str)
d = d[d.text.astype(str).str.len() > 10].drop_duplicates("text")   # exact duplicates across splits = leakage
print(d.label.value_counts())
# sample FIRST (cleaning 1M rows is slow), then clean only the sample
d = pd.concat([g.sample(min(len(g), int(N_PER_CLASS * 1.2)), random_state=SEED) for _, g in d.groupby("label")])
d["text"] = d["text"].astype(str).map(clean_urdu).str.slice(0, 2000)   # 2000 chars is plenty for MAX_LEN tokens
d = d[d.text.str.len() > 10].drop_duplicates("text")
d = pd.concat([g.head(N_PER_CLASS) for _, g in d.groupby("label")])

classes = sorted(d.label.unique())
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}
d["y"] = d.label.map(label2id)

train_df, tmp = train_test_split(d, test_size=0.2, stratify=d.y, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp.y, random_state=SEED)
for n, x in [("train", train_df), ("val", val_df), ("test", test_df)]:
    x.to_csv(f"{OUT}/{n}.csv", index=False)                 # keep splits for reproducibility
print(len(train_df), len(val_df), len(test_df)); print(d.label.value_counts())

## Section 6 — Helpers + TF-IDF baseline
Needs: Sections 2, 5. Defines: scores, results, save_results.

In [ ]:
print('=== Section 6: Helpers + TF-IDF baseline ===')
# ---- helpers + classical baseline ----
def scores(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    fw = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[2]
    return {"accuracy": accuracy_score(y_true, y_pred), "precision_macro": p,
            "recall_macro": r, "f1_macro": f, "f1_weighted": fw}

RESULTS_FILE = f"{OUT}/results.json"
results = json.load(open(RESULTS_FILE)) if os.path.exists(RESULTS_FILE) else {}
def save_results():
    json.dump(results, open(RESULTS_FILE, "w"), indent=2, ensure_ascii=False)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
t0 = time.time()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
clf = LinearSVC().fit(vec.fit_transform(train_df.text), train_df.y)
pred = clf.predict(vec.transform(test_df.text))
results["TFIDF+LinearSVC"] = {**scores(test_df.y, pred), "train_time_s": time.time() - t0}
save_results(); print(results["TFIDF+LinearSVC"])

## Section 7 — Token fertility
Needs: Sections 2, 5, 6.

In [ ]:
print('=== Section 7: Token fertility ===')
# ---- tokenizer "fertility" (tokens per word): how well each tokenizer handles Urdu ----
def fertility(hf_id, n=500):
    tok = AutoTokenizer.from_pretrained(hf_id)
    s = train_df.text.sample(min(n, len(train_df)), random_state=SEED).tolist()
    return sum(len(tok(x, add_special_tokens=False).input_ids) for x in s) / sum(len(x.split()) for x in s)

results["_token_fertility"] = {name: fertility(hf) for name, hf in MODELS.items()}
save_results(); print(results["_token_fertility"])

## Section 8 — Define run_experiment()
Needs: Sections 2, 5, 6. Only defines the function (prints just the banner).

In [ ]:
print('=== Section 8: Define run_experiment() ===')
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def run_experiment(name, hf_id, method, rank=RANK):
    tag = f"{name}_{method}_r{rank}"
    if tag in results:
        print("skip", tag); return
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    tok = AutoTokenizer.from_pretrained(hf_id)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    def enc(x):
        ds = Dataset.from_dict({"text": x.text.tolist(), "labels": x.y.tolist()})
        return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                      batched=True, remove_columns=["text"])
    tr, va, te = enc(train_df), enc(val_df), enc(test_df)

    kw = dict(num_labels=len(classes), id2label=id2label, label2id=label2id)
    if "gemma" in hf_id.lower(): kw["attn_implementation"] = "eager"   # recommended for training Gemma 2
    if method == "qlora":
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForSequenceClassification.from_pretrained(
            hf_id, quantization_config=bnb, device_map={"": 0}, **kw)
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    else:   # lora / dora: fp16 base (T4 has no bf16)
        model = AutoModelForSequenceClassification.from_pretrained(
            hf_id, dtype=torch.float16, device_map={"": 0}, **kw)
    model.config.pad_token_id = tok.pad_token_id
    model.config.use_cache = False

    cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=2 * rank, lora_dropout=0.05,
                     target_modules=TARGETS, use_dora=(method == "dora"))
    model = get_peft_model(model, cfg)
    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()          # fp16 AMP needs fp32 trainable weights
    trainable, total = model.get_nb_trainable_parameters()

    args = TrainingArguments(
        output_dir=f"{OUT}/ckpt/{tag}", num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=8, per_device_eval_batch_size=16, gradient_accumulation_steps=2,
        lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.01, fp16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit" if method == "qlora" else "adamw_torch",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        logging_steps=5 if DRY_RUN else 50, report_to="none", seed=SEED)

    def compute_metrics(p):
        logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        return scores(p.label_ids, logits.argmax(-1))

    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    t0 = time.time(); trainer.train(); train_time = time.time() - t0

    out = trainer.predict(te)
    logits = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
    pred = logits.argmax(-1)
    np.save(f"{OUT}/preds_{tag}.npy", pred)                  # needed later for McNemar / confusion matrix
    results[tag] = {
        **scores(test_df.y.values, pred),
        "trainable_params": int(trainable), "trainable_pct": 100 * trainable / total,
        "train_time_s": train_time, "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
        "test_samples_per_s": out.metrics.get("test_samples_per_second"),
        "per_class": classification_report(test_df.y.values, pred, target_names=classes,
                                           output_dict=True, zero_division=0),
        "history": trainer.state.log_history,
    }
    save_results(); print(tag, {k: round(v, 4) for k, v in results[tag].items() if k in ("accuracy", "f1_macro")})
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
   results.pop("Gemma-2-2B_lora_r16", None); save_results()

## Section 9 — Run experiments (training)
Needs: Sections 2, 5, 6, 8. This is the long GPU step.

In [ ]:
print('=== Section 9: Run experiments (training) ===')
for method in METHODS:
    for name, hf_id in MODELS.items():
        try:
            run_experiment(name, hf_id, method)
        except Exception as e:
            import traceback; traceback.print_exc()
            print("FAILED", name, method)
        gc.collect(); torch.cuda.empty_cache()

## Section 10 — Summary table
Needs: Sections 6, 9. Defines: summary.

In [ ]:
print('=== Section 10: Summary table ===')
cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted",
        "trainable_pct", "train_time_s", "peak_vram_gb"]
rows = [{"model": k, **{c: v[c] for c in cols if c in v}}
        for k, v in results.items() if isinstance(v, dict) and "accuracy" in v]
summary = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
summary.to_csv(f"{OUT}/summary.csv", index=False)
summary.round(4)

## Section 11 — Plots (F1 bar + confusion matrix)
Needs: Sections 5, 10.

In [ ]:
print('=== Section 11: Plots (F1 bar + confusion matrix) ===')
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ax = summary.plot.barh(x="model", y="f1_macro", legend=False, figsize=(8, 5)); ax.invert_yaxis()
plt.xlabel("Macro-F1"); plt.tight_layout(); plt.savefig(f"{OUT}/f1_bar.png", dpi=300); plt.show()

best = summary[summary["model"].str.contains("_r")].iloc[0]["model"]    # best PEFT run
ConfusionMatrixDisplay.from_predictions(test_df.y, np.load(f"{OUT}/preds_{best}.npy"),
                                        display_labels=classes, xticks_rotation=45)
plt.tight_layout(); plt.savefig(f"{OUT}/cm_{best}.png", dpi=300); plt.show()